In [ ]:
import numpy as np

In [ ]:
class LinearRegression:

    #Setting random w and b
    def __init__(self):
        self.w = np.random.randn(1)
        self.b = np.random.randn(1)

    #this is forward pass
    def forward(self, x):
        return self.w*x+self.b

    def gradients(self, x, y, y_pred):
        n = len(x)
        error = y - y_pred
        dw = (-2 / n) * np.sum(x * error)
        db = (-2 / n) * np.sum(error)

        return dw, db

In [ ]:
def generate_data(n_samples=500, noise_std=0.5, random_seed=42):
    """
    Generate synthetic data using:
    y = wx + b + noise
    """

    np.random.seed(random_seed)
    x = np.random.randn(n_samples, 1)

    noise = np.random.normal(
        loc=0.0,
        scale=noise_std,
        size=(n_samples, 1)
    )

    y = 3 * x + 2 + noise

    return x, y

In [ ]:
def mse_loss(y_true, y_pred):

    # MSE
    loss = np.mean((y_true - y_pred) ** 2)

    return loss

In [ ]:
class Momentum:

    def __init__(self, learning_rate=0.01, beta=0.9):

        self.lr = learning_rate
        self.beta = beta

        self.vw = 0
        self.vb = 0

    def step(self, model, dw, db):

        self.vw = self.beta * self.vw + (1 - self.beta) * dw
        self.vb = self.beta * self.vb + (1 - self.beta) * db

        model.w -= self.lr * self.vw
        model.b -= self.lr * self.vb

In [ ]:
class Adam:

    def __init__(self,
                 learning_rate=0.001,
                 beta1=0.9,
                 beta2=0.999,
                 epsilon=1e-8):
        
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon

        self.mw = 0
        self.mb = 0

        self.vw = 0
        self.vb = 0

        self.t = 0

    def step(self, model, dw, db):

        self.t += 1

        #First moment
        self.mw = self.beta1 * self.mw + (1 - self.beta1) * dw
        self.mb = self.beta1 * self.mb + (1 - self.beta1) * db

        #Second moment
        self.vw = self.beta2 * self.vw + (1 - self.beta2) * (dw ** 2)
        self.vb = self.beta2 * self.vb + (1 - self.beta2) * (db ** 2)

        # Bias correction
        mw_hat = self.mw / (1 - self.beta1 ** self.t)
        mb_hat = self.mb / (1 - self.beta1 ** self.t)

        vw_hat = self.vw / (1 - self.beta2 ** self.t)
        vb_hat = self.vb / (1 - self.beta2 ** self.t)

        # Parameter update
        model.w -= self.lr * mw_hat / (np.sqrt(vw_hat) + self.epsilon)
        model.b -= self.lr * mb_hat / (np.sqrt(vb_hat) + self.epsilon)

In [ ]:
def train(model,
          optimizer,
          x,
          y,
          epochs=100):

    history = []

    for epoch in range(epochs):

        for i in range(len(x)):

            x_i = x[i:i+1]
            y_i = y[i:i+1]

            y_pred = model.forward(x_i)

            dw, db = model.gradients(
                x_i,
                y_i,
                y_pred
            )

            optimizer.step(model, dw, db)

        loss = mse_loss(
            y,
            model.forward(x)
        )

        history.append(loss)

    return history

In [ ]:
import matplotlib.pyplot as plt

def main():

    # Dataset
    x, y = generate_data()

    epochs = 100

    # Save identical initialization
    np.random.seed(42)

    initial_model = LinearRegression()

    initial_w = initial_model.w.copy()
    initial_b = initial_model.b.copy()

    histories = {}

    optimizers = {
        "SGD": SGD(learning_rate=0.01),
        "Momentum": Momentum(learning_rate=0.01, beta=0.9),
        "Adam": Adam(learning_rate=0.01)
    }

    for name, optimizer in optimizers.items():

        model = LinearRegression()

        model.w = initial_w.copy()
        model.b = initial_b.copy()

        history = train(
            model,
            optimizer,
            x,
            y,
            epochs
        )

        histories[name] = history

        print(f"{name}")
        print(f"Final Weight : {model.w}")
        print(f"Final Bias   : {model.b}")
        print(f"Final Loss   : {history[-1]:.6f}")
        print("-" * 40)

    # Plot comparison
    plt.figure(figsize=(10, 6))

    for name, history in histories.items():
        plt.plot(history, label=name)

    plt.xlabel("Epoch")
    plt.ylabel("Mean Squared Error")
    plt.title("SGD vs Momentum vs Adam")

    plt.grid(True)
    plt.legend()

    plt.show()

In [ ]:
main()